In [57]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_core.stores import InMemoryStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.retrievers.document_compressors import cohere_rerank
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder
from langsmith import traceable
from langchain.chat_models import init_chat_model
from langchain_huggingface import HuggingFaceEmbeddings
from pydantic import BaseModel, field_validator
from dotenv import load_dotenv
from torch import embedding
# from all_document.data import documents
import os
from langchain_classic.document_loaders import DirectoryLoader, TextLoader, PyMuPDFLoader
from langchain_chroma import Chroma

In [2]:
import textwrap
def wrap_text(text, width=90):
    #split the input text into line based on newline characters
    line = text.split('\n')

    #wrap each line individual
    wrapped_line = [textwrap.fill(l, width) for l in line]

    #join wrapper line back together using newline characters
    wrapped_text = '\n'.join(wrapped_line)

    return wrapped_text

In [ ]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

encode_kwargs = {'normalize_embeddings' : True}
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                        encode_kwargs=encode_kwargs)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4541.87it/s]


In [ ]:
data_path = "G:\\langc-in-production\\all_document\\ai_agent.pdf"

loader = DirectoryLoader(path=data_path,
                         show_progress=True,
                         loader_cls=PyMuPDFLoader, 
                         glob="**/*.pdf")
docs = loader.load()

print(len(docs))

1731


In [14]:
from os import path


loader = PyMuPDFLoader(data_path)

docss = loader.load()

In [15]:
print(len(docss))

288


In [20]:
doc = docss[:100]

In [24]:
print(doc[30])

page_content='Fundamentals of Generative AI
6
game elements such as character design, level layouts, music and sound effects, and so on. By 
providing different conditions such as “Create a forest level” or “Create a desert level,” the CVAE 
can produce a wide variety of game environments, saving time for designers and enhancing 
the player’s experience with more diverse and interesting game worlds [3].
GANs
A GAN is basically formed by two neural networks: a generator and a discriminator. The generator 
generates synthetic data samples; the other trained neural network should then be able to tell the 
difference between real and created samples. While training these networks, they are trained together 
antagonistically: the generator tries to fool the discriminator, while the discriminator tries rightly to 
classify real versus fake data. In this competition, the generator gets better and better at faking data. 
The following are some of the different types of GANs:
•	 GAN: The basic 

In [30]:
print(doc[90].page_content)

Essential Components of Intelligent Agents
66
•	 Optimal path finding: This is a specific type of graph search that aims to find not just any path, 
but the best path according to some criteria (usually minimizing total edge cost). Two of the 
algorithms in this category are the Bellman-Ford algorithm and the A* search.
The downsides of using graph-based planning algorithms include fixing the state representation (state 
space) upfront, and the potential for exponential growth in the number of states to represent and 
store as problems get more complex.
Graph-based planning techniques find numerous real-world applications across domains, where 
finding optimal sequences of actions to achieve goals is crucial. These applications include navigation 
and route planning, such as GPS systems using graph representations of road networks to find optimal 
routes minimizing travel time or distance. Logistics and supply chain applications involve planning 
optimal sequences of operations for man

In [31]:
raw_data = ''

for i, doc in enumerate(doc):
    text = doc.page_content
    if text:
        raw_data += text

In [32]:
raw_data

'Building Agentic AI Systems\nCreate intelligent, autonomous AI agents that can reason, plan, \nand adapt\nAnjanava Biswas\nWrick TalukdarBuilding Agentic AI Systems\nCopyright © 2025 Packt Publishing\nAll rights reserved. No part of this book may be reproduced, stored in a retrieval system, or transmitted \nin any form or by any means, without the prior written permission of the publisher, except in the case \nof brief quotations embedded in critical articles or reviews.\nThe author acknowledges the use of cutting-edge AI, such as ChatGPT, with the sole aim of enhancing \nthe language and clarity within the book, thereby ensuring a smooth reading experience for readers. \nIt’s important to note that the content itself has been crafted by the author and edited by a professional \npublishing team.\nEvery effort has been made in the preparation of this book to ensure the accuracy of the information \npresented. However, the information contained in this book is sold without warranty, eit

In [37]:
len(raw_data)

195454

In [33]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
    length_function = len,
    is_separator_regex= False
)

In [35]:
texts = text_splitter.split_text(raw_data)

In [36]:
len(texts)

496

In [39]:
print(texts[100])

to think about what creativity is, who the artist really is, and what the ethical parameters should be 
for AI-created content.
Having understood what generative AI is and its brief history, let’s explore the different types of 
generative AI models.
Types of generative AI models
Generative AI is an exciting domain of AI that deals with the generation of new, synthetic data by 
learning patterns from existing datasets, aiming to generate outputs that share similar statistical


In [42]:
db = Chroma.from_texts(texts=texts, embedding=embedding_model, persist_directory="./chroma_db")

In [44]:
query = "what is agentic ai"

data = db.similarity_search(query=query, k =5)

In [52]:
for i, q in enumerate(data, start=1):
    print(f"Query_Ranked:{i} = {q.page_content}\n")

Query_Ranked:1 = on experience
Start building agentic AI
We have learned quite a lot about the characteristics of intelligent agents, how they are built, how they 
work with different algorithms, and their essential components. It is now time for a gentle introduction 
to the world of agentic AI and to start building applications using different frameworks.
In subsequent chapters of this book, we will make extensive use of several open source frameworks. The

Query_Ranked:2 = You can find the code files for this chapter at https://github.com/PacktPublishing/
Building-Agentic-AI-Systems and follow the README file in the repository to set up your 
development environment.
Understanding self-governance, agency, and autonomy
The captivating aspect of agentic systems lies in the intricate decision-making processes they employ, 
which provide valuable insights into how choices are optimized within specific contexts. These systems

Query_Ranked:3 = Agentic AI systems have been applied in doma

### Retrieval Setup

In [55]:
retrievers = db.as_retriever(k=3)

retrievers.invoke(query)

[Document(id='a8a3f417-4499-4282-a288-3f0092ff77e1', metadata={}, page_content='on experience\nStart building agentic AI\nWe have learned quite a lot about the characteristics of intelligent agents, how they are built, how they \nwork with different algorithms, and their essential components. It is now time for a gentle introduction \nto the world of agentic AI and to start building applications using different frameworks.\nIn subsequent chapters of this book, we will make extensive use of several open source frameworks. The'),
 Document(id='3d078e28-6742-48c7-bcd3-bad45078ee0a', metadata={}, page_content='You can find the code files for this chapter at https://github.com/PacktPublishing/\nBuilding-Agentic-AI-Systems and follow the README file in the repository to set up your \ndevelopment environment.\nUnderstanding self-governance, agency, and autonomy\nThe captivating aspect of agentic systems lies in the intricate decision-making processes they employ, \nwhich provide valuable insi

### Chat Chain

In [58]:
template = """Answer the following question based only from the context:
{context}

Question: {question}
"""

In [60]:
prompt = ChatPromptTemplate.from_template(template)

In [61]:
llm = init_chat_model("google_genai:gemini-2.5-flash")

In [62]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the following question based only from the context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [63]:
llm

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x0000018000C70180>, default_metadata=(), model_kwargs={})

In [64]:
chain = (
    {"context" : retrievers, "question" : RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [66]:
text_reply = chain.invoke("how to swimm")

print(wrap_text(text_reply))

I apologize, but the provided context does not contain any information on how to swim. The
documents discuss AI workflows, reinforcement learning in robotics, LLMs, and agentic
systems.


In [67]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.prompts import ChatMessagePromptTemplate, PromptTemplate

In [ ]:
from langchain_core import messages


prompt = ChatPromptTemplate(input_variables=['original_query'],
                            messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='you are a helpful assistant that generates multiples search query on a single input query.')),
                                     HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['original_query'], template='generate muliple search queries related to : {question} \n Output (4 queries):'))])


TypeError: ChatPromptTemplate.__init__() missing 1 required positional argument: 'messages'